In [ ]:
import os
import pandas as pd

In [ ]:
def count_element_in_xyz(file_path, element_symbol):
    '''
    Count the number of the specified element
    Args:
    file_path: The input xyz file
    Element_symbol: The element that needs to be calculated

    Return:The counting number of the specified element
    '''

    element_count = 0
    
    with open(file_path, 'r') as file:
        lines = file.readlines()
        
        # Skip the first two lines
        for line in lines[2:]:
            parts = line.split()
            if len(parts) == 4:  
                element = parts[0]
                if element == element_symbol:
                    element_count += 1
    
    return element_count

def count_unit_in_xyz(folder_path):
    '''
    Count the number of SBUs and BTBs in MOL respectively
    Args:
    file_path: The folder path that contains all of the xyz files to be processed
    Element_symbol: The element that needs to be calculated

    Return: The CSV file storing the final statistical results 
    '''
    
    results = {}
    
    for file_name in os.listdir(folder_path):
        if file_name.startswith('assembly') and file_name.endswith('.xyz'):
            file_path = os.path.join(folder_path, file_name)
            
            # Calculate the number of Hf
            Zr_count = count_element_in_xyz(file_path, 'Zr')
            SBU_count = Zr_count / 6
            
            # Calculate the number of C
            C_count = count_element_in_xyz(file_path, 'C')
            BTB_count = (C_count - 12 * SBU_count) / 27
            
            results[file_name] = {'SBU': SBU_count, 'BTB': BTB_count}
    
    return results

folder_path = os.getcwd()  # get the current directory
results = count_unit_in_xyz(folder_path)

# Convert results to DataFrame and save to CSV
df = pd.DataFrame.from_dict(results, orient='index')
df.index.name = 'File Name'
df.to_csv('results.csv')

print("Results saved to results.csv")

In [ ]:
def process_xyz_file(input_file, output_file):
    '''
    Replace the element label "Zr" with "Hf" and remove duplicate coordinates.
    Args:
        input_file: The original xyz file
        output_file: The processed xyz file
    '''
    # Remove the duplicate coordinates
    unique_coords = set()
    unique_lines = []

    with open(input_file, 'r') as infile:
        lines = infile.readlines()
        
        # Skip the first two lines
        num_atoms = int(lines[0].strip())
        comment = lines[1].strip()
        
        for line in lines[2:]:
            parts = line.split()
            if len(parts) == 4:
                element, x, y, z = parts
                # Replace "Zr" with "Hf"
                if element == 'Zr':
                    element = 'Hf'
                coord = (element, float(x), float(y), float(z))
                if coord not in unique_coords:
                    unique_coords.add(coord)
                    unique_lines.append(f"{element} {x} {y} {z}\n")
        
        # Update the number of atoms
        num_atoms = len(unique_lines)

    # Write the output file
    with open(output_file, 'w') as outfile:
        outfile.write(f"{num_atoms}\n")
        outfile.write(f"{comment}\n")
        outfile.writelines(unique_lines)

def process_all_assembly_files():
    '''
    Process all xyz files starting with "assembly" in the current directory.
    '''
    # Get the current working directory
    current_directory = os.getcwd()
    
    # Traverse all files in the current directory
    for file_name in os.listdir(current_directory):
        # Check if the file starts with "assembly" and ends with ".xyz"
        if file_name.startswith('assembly') and file_name.endswith('.xyz'):
            input_file = os.path.join(current_directory, file_name)
            # Generate the output file name by appending "_processed" before the extension
            output_file = os.path.join(current_directory, file_name)
            # Process the file
            process_xyz_file(input_file, output_file)
            print(f"Processed file saved as: {output_file}")

# Run the main program
process_all_assembly_files()